# Testing different Classifier heads

In [1]:
import sys
from pathlib import Path
# fixing imports for src/ modules
sys.path.append(str(Path().resolve().parents[1]))

import copy
import pprint

import torch
import torch.nn as nn
import src.cnn.cnn_paths as paths
import src.cnn.cnn_models as cnn_models
from src.cnn.configs import ExperimentConfig
from src.cnn.cnn_registry import build_model
from src.cnn.cnn_utils import (
    load_datasets,
    print_dataset_info,
    make_train_loader,
    make_eval_loader,
    get_device,
    get_num_parameters,
    train_eval,
    evaluate, run_experiment,
)

In [ ]:
# test header size
fc_options = [1, 2, 3, 4, 5]

all_results = {}

for fc in fc_options:
    print(f"\n==== Training with FC layers = {fc} ====\n")

    # see configs.py
    cfg = ExperimentConfig()

    # =========================================================
    # DATASET CONFIG
    # =========================================================
    cfg.dataset.dataset_dir = paths.DATASET_DIR # see cnn_paths.py
    cfg.dataset.train_subdir = "train"
    cfg.dataset.val_subdir = "validate"
    cfg.dataset.test_subdir = None   # or "test" if you have it
    cfg.dataset.image_size = 224

    # leave transforms as None -> default Resize + ToTensor pipeline
    cfg.dataset.train_transform = None
    cfg.dataset.eval_transform = None

    # optional normalization
    cfg.dataset.normalize_mean = None
    cfg.dataset.normalize_std = None


    # =========================================================
    # MODEL CONFIG
    # IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
    # use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
    # =========================================================
    cfg.model.name = "depth_cnn_fc"
    cfg.model.kwargs = {
        "depth": 12,
        "in_channels": 3,
        "num_classes": 10,
        "base_channels": 32,
        "max_channels": 256,
        "dropout_conv": 0.0,
        "dropout_fc": 0.5,
        "inputsize": 224,
        "fc_layers": fc,
        # construction decreasing header to keep numbers of parameters in a reasonable range
        "hidden1": 128,
        "hidden2": 64,
        "hidden3": 32,
        "hidden4": 16,
    }


    # =========================================================
    # DATALOADER CONFIG
    # =========================================================
    cfg.loader.batch_size = 32 # reduced batchsize from learnings in previous experiments
    cfg.loader.num_workers = 0
    cfg.loader.pin_memory = True
    cfg.loader.train_shuffle = True
    cfg.loader.eval_shuffle = False
    cfg.loader.drop_last_train = False
    cfg.loader.drop_last_eval = False


    # =========================================================
    # LOSS CONFIG
    # =========================================================
    cfg.loss.cls = nn.CrossEntropyLoss
    cfg.loss.kwargs = {}


    # =========================================================
    # OPTIMIZER CONFIG
    # =========================================================
    cfg.optimizer.cls = torch.optim.SGD
    cfg.optimizer.kwargs = {
        "lr": 0.01, # from previous testings
        "momentum": 0.9, # default value for momentum
        "weight_decay": 1e-4, # deafault value
    }


    # =========================================================
    # SCHEDULER CONFIG
    # Example: Reduce LR when validation loss plateaus
    # =========================================================
    cfg.scheduler.cls = None
    cfg.scheduler.kwargs = {
        # "mode": "min",
        # "factor": 0.5,
        # "patience": 2,
    }
    cfg.scheduler.step_metric = None


    # =========================================================
    # TRAIN CONFIG
    # =========================================================
    cfg.train.epochs = 100
    cfg.train.device = str(get_device("auto"))
    cfg.train.non_blocking = True
    cfg.train.use_amp = (cfg.train.device == "cuda") # suggestion from ChatGPT for large batch sizes to save memory and speed up training
    cfg.train.grad_clip_norm = None
    cfg.train.best_metric = "val/accuracy"
    cfg.train.best_mode = "max"
    cfg.train.seed = 42


    # =========================================================
    # W&B CONFIG
    # =========================================================
    cfg.wandb.enabled = True
    cfg.wandb.project = "MPW-CNN"
    cfg.wandb.entity = "MSE_DeLearn_SPR26"
    cfg.wandb.mode = "online"   # "online", "offline", or "disabled" for no logging

    # NOTE: use meaningful names for runs, so the difference is clear
    cfg.wandb.run_name = f"BatchSize_32_depth{cfg.model.kwargs['depth']}_momentum{cfg.optimizer.kwargs['momentum']}_lr{cfg.optimizer.kwargs['lr']}_fc_layers{cfg.model.kwargs['fc_layers']}_e{cfg.train.epochs}"

    # NOTE: use meaningful grouping, for example, by model or by task.
    # E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
    cfg.wandb.group = "header_fc_layer_influence"
    cfg.wandb.job_type = "train"

    # NOTE: use meaningful tags to filter runs in UI
    cfg.wandb.tags = ["deep_cnn", "fc_layers"]
    cfg.wandb.notes = "Full config live run"

    cfg.wandb.log_epoch_metrics = True
    cfg.wandb.log_every_n_epochs = 1

    cfg.wandb.metric_allowlist = {
        "train/loss",
        "train/accuracy",
        "val/loss",
        "val/accuracy",
        "gap/accuracy",
        "gap/loss",
        "lr",
    }

    cfg.wandb.summary_allowlist = {
        "best_epoch",
        "best_metric_name",
        "best_metric_value",
        "train_final/loss",
        "train_final/accuracy",
        "val_final/loss",
        "val_final/accuracy",
    }

    cfg.wandb.watch_model = False
    cfg.wandb.watch_log = "all"
    cfg.wandb.watch_log_freq = 100

    # ---------------------------
    # RUN
    # ---------------------------
    # with high batchsizes GPU memory may not be sufficient, so we catch OOM errors and skip those runs
    try:
        model, history, result = run_experiment(cfg)
        all_results[fc] = copy.deepcopy(result)

        # give some time for GPU memory to be freed before next run
        del model, history, result
        torch.cuda.empty_cache()

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM at batch size {fc}, skipping...")
            torch.cuda.empty_cache()
            break # if we get OOM at a certain batch size, it's likely that all larger batch sizes will also be OOM, so we can break the loop early
        else:
            raise

# ---------------------------------------------------------
# summary
# ---------------------------------------------------------
print("\n" + "="*80)
print("SUMMARY (FC Layer Comparison)")
print("="*80)

for fc, res in all_results.items():
    print(
        f"fc={fc:>3} | "
        f"best_val_acc={res['best_metric_value']:.4f} | "
        f"best_epoch={res['best_epoch']}"
    )

